**HW 2/2.** Upgrade the BSM call option pricer to be able to return the greeks as well, not just the price.

**Professor's code** (cells 43 and 44 of the notebook)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from typing import Dict, List, Union

"""The pricing function of European call option"""
def black_scholes_eur_call(r: float, T: float, S0: float, sigma: float, K: Union[float, List[float], np.ndarray]) -> np.ndarray:
    """
    Black-Scholes pricer of European call option on non-dividend-paying stock

    param r: risk-free interest rate (which is constant)
    param T: time to maturity (in years)
    param S0: initial spot price of the underlying stock
    param sigma: volatility of the underlying stock
    param K: strike price (or prices)
    """
    # check conditions
    assert sigma > 0

    K = np.array([K]) if isinstance(K, float) else np.array(K)

    d1_vec = ( np.log( S0 / K ) + ( r + 0.5 * sigma**2 ) * T ) / ( sigma * T**0.5 )
    d2_vec = d1_vec - sigma * T**0.5

    N_d1_vec = norm.cdf(d1_vec)
    N_d2_vec = norm.cdf(d2_vec)

    return N_d1_vec * S0 - K_vec * np.exp((-1.0)*r*T) * N_d2_vec

In [ ]:
# Strike prices for which the option price is calculated
K_vec = np.arange(10, 30, 0.01)

# Time to maturities (in year fractions)
T_vec = [1.0, 2.0, 5.0]

prices_to_plot = []
for _T in T_vec:
    prices_to_plot.append(black_scholes_eur_call(r=0.05, T=_T, S0=20.0, sigma=0.3, K=K_vec))

plt.figure(figsize=(10, 6))

for prices, T in zip(prices_to_plot, T_vec):
    plt.plot(K_vec, prices, label=f"T = {int(T)}")

plt.title(
    'Price of a European Call ($S_0 = 20$, $\\sigma = 0.3$, $r = 0.05$)\n\n'
    '$T$: time to maturity in years'
)

plt.xlabel('Strike price $K$')
plt.ylabel('Call option price $C(S_0,K,T)$')

plt.legend()
plt.grid(True)
plt.show()

**Test of the professor's code.** The return line uses `K_vec` (global variable of cell 44), not the argument `K`: one strike gives 2000 numbers.

In [ ]:
# test: one strike K = 20.0 should give one price
c = black_scholes_eur_call(r=0.05, T=1.0, S0=20.0, sigma=0.3, K=20.0)
print(c.shape)
print(c[:3])

**NEW VERSION**

- change # 1: the return line uses the argument `K`; the price is stored in `price`.
- change # 2: new argument `return_greeks` (default `False`: same output as before). With `True` it returns a dict with the price and the greeks of cell 49 (with t = 0).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from typing import Dict, List, Union

"""The pricing function of European call option"""
# change # 2
#Delete.    def black_scholes_eur_call(r: float, T: float, S0: float, sigma: float,
#Delete.                               K: Union[float, List[float], np.ndarray]) -> np.ndarray:
# replace by
def black_scholes_eur_call(r: float, T: float, S0: float, sigma: float,
                           K: Union[float, List[float], np.ndarray],
                           return_greeks: bool = False):
# end change # 2
    """
    Black-Scholes pricer of European call option on non-dividend-paying stock

    param r: risk-free interest rate (which is constant)
    param T: time to maturity (in years)
    param S0: initial spot price of the underlying stock
    param sigma: volatility of the underlying stock
    param K: strike price (or prices)
    param return_greeks: if True, return a dict with the price and the greeks
    """
    # check conditions
    assert sigma > 0

    K = np.array([K]) if isinstance(K, float) else np.array(K)

    d1_vec = ( np.log( S0 / K ) + ( r + 0.5 * sigma**2 ) * T ) / ( sigma * T**0.5 )
    d2_vec = d1_vec - sigma * T**0.5

    N_d1_vec = norm.cdf(d1_vec)
    N_d2_vec = norm.cdf(d2_vec)

    # change # 1
    #Delete.    return N_d1_vec * S0 - K_vec * np.exp((-1.0)*r*T) * N_d2_vec
    # replace by
    price = N_d1_vec * S0 - K * np.exp((-1.0)*r*T) * N_d2_vec
    # end change # 1

    # change # 2
    if not return_greeks:
        return price
    n_d1_vec = norm.pdf(d1_vec)                  # N'(d1)
    return {
        "price": price,
        "delta": N_d1_vec,
        "gamma": n_d1_vec / (S0 * sigma * T**0.5),
        "vega":  S0 * n_d1_vec * T**0.5,
        "theta": -S0 * n_d1_vec * sigma / (2 * T**0.5) - r * K * np.exp(-r*T) * N_d2_vec,
        "rho":   K * T * np.exp(-r*T) * N_d2_vec,
    }
    # end change # 2

In [ ]:
# test 1: one strike gives one price
print(black_scholes_eur_call(r=0.05, T=1.0, S0=20.0, sigma=0.3, K=20.0))

# test 2: price and greeks, S0 = K = 100, r = 0.05, sigma = 0.2, T = 1
g = black_scholes_eur_call(r=0.05, T=1.0, S0=100.0, sigma=0.2, K=100.0,
                           return_greeks=True)
for name, value in g.items():
    print(name, round(float(value[0]), 6))

Cell 44 runs unchanged with the new version (same figure).

In [ ]:
# Strike prices for which the option price is calculated
K_vec = np.arange(10, 30, 0.01)

# Time to maturities (in year fractions)
T_vec = [1.0, 2.0, 5.0]

prices_to_plot = []
for _T in T_vec:
    prices_to_plot.append(black_scholes_eur_call(r=0.05, T=_T, S0=20.0, sigma=0.3, K=K_vec))

plt.figure(figsize=(10, 6))

for prices, T in zip(prices_to_plot, T_vec):
    plt.plot(K_vec, prices, label=f"T = {int(T)}")

plt.title(
    'Price of a European Call ($S_0 = 20$, $\\sigma = 0.3$, $r = 0.05$)\n\n'
    '$T$: time to maturity in years'
)

plt.xlabel('Strike price $K$')
plt.ylabel('Call option price $C(S_0,K,T)$')

plt.legend()
plt.grid(True)
plt.show()